<img src="https://raw.githubusercontent.com/iamYannC/Ebbinghaus/main/docs/hex.png" alt="Ebbinghaus hex logo" align="right" width="15%" height="auto"/>

The **Ebbinghaus Illusion Benchmark** is designed to evaluate both VLMs and human participants on variants of the Ebbinghaus illusion.

This notebook is a Kaggle implementation of the original project, using [Gemma 3 4B](https://huggingface.co/google/gemma-3-4b-it) via [Hugging Face](https://huggingface.co/) (requires an API key) on Kaggle's free GPU runtime. The pipeline is identical to the original, with minor adaptations for this environment.

The full code (R and Python) and technical reference are available at [`iamYannC/Ebbinghaus`](https://github.com/iamYannC/Ebbinghaus).

_© [Yann Cohen](https://orcid.org/0009-0009-0509-3609) 2026 - CC BY 4.0 - [doi](https://doi.org/10.5281/zenodo.18906801)_
___


# Preparations

## Imports
All Python dependencies used by this notebook.

In [ ]:
import os
import sys
import time
from datetime import datetime, timezone

import pandas as pd
import torch
from PIL import Image
from transformers import AutoProcessor, Gemma3ForConditionalGeneration

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

## Setup
Configure filesystem paths, create output directories, and authenticate with Hugging Face.

This step requires a (free) Hugging Face API key and registering it as a secret on Kaggle.

**DO NOT hardcode the key in this notebook!**

In [ ]:
INPUT = "/kaggle/input/datasets/yanncohen/ebbinghaus-illusion-benchmark"
WORK  = "/kaggle/working"

IMAGES      = f"{WORK}/output/images"
IMAGES_EVAL = f"{WORK}/output/images_eval"
TRIALS_CSV  = f"{WORK}/output/trials.csv"

os.makedirs(IMAGES,      exist_ok=True)
os.makedirs(IMAGES_EVAL, exist_ok=True)

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

# Phase 1

## Generate Data
build a design matrix, render stimulus images, and load prompt templates.

In [ ]:
sys.path.insert(0, f"{INPUT}/py")
from src.generate_design import generate_design
from src.render_stimuli import render_stimuli
from src.evaluate import fill_prompt, parse_response

trials = generate_design(seed=42, n_per_tier=5, image_dir=IMAGES)
render_stimuli(trials)
trials.to_csv(TRIALS_CSV, index=False)

prompts = pd.read_csv(f"{INPUT}/data/prompts.csv")
print(f"Generated {len(trials)} trials, loaded {len(prompts)} prompts")

## Prepare Evaluation Images
Copy stimulus images with ground-truth info stripped from filenames, so the model can't cheat.

In [ ]:
from src.strip_answer import strip_answer_from_images

trials = strip_answer_from_images(trials, output_dir=IMAGES_EVAL)

Copied 20 images to /kaggle/working/images_eval with answer-stripped filenames.


# Phase 2

## Load Model
Load the Gemma 3 4B-IT model and processor onto GPU.

In [ ]:
model_id  = "google/gemma-3-4b-it"
processor = AutoProcessor.from_pretrained(model_id)
model     = Gemma3ForConditionalGeneration.from_pretrained(
                model_id, device_map="cuda", torch_dtype=torch.bfloat16
            )

## Run Evaluation
Loop over every (prompt × trial) combination: build a chat message with the stimulus image, generate a response, parse the answer, and record the result.

In [ ]:
results = []

for _, prompt_row in prompts.iterrows():
    prompt_id       = prompt_row["prompt_id"]
    prompt_template = prompt_row["user_prompt_template"]
    response_format = prompt_row["response_format"]

    for _, row in trials.iterrows():
        trial = row.to_dict()
        raw_text = fill_prompt(prompt_template, trial)

        messages = [
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": raw_text}]}
        ]

        chat_prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
        image  = Image.open(row["file_path"])
        inputs = processor(
            images=image,
            text=chat_prompt,
            return_tensors="pt"
        ).to(model.device)

        t0 = time.time()
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=50)
        latency_ms = round((time.time() - t0) * 1000)

        input_len = inputs["input_ids"].shape[-1]
        response  = processor.decode(output[0][input_len:], skip_special_tokens=True).strip().lower()
        parsed    = parse_response(response, row["orientation"], response_format)

        results.append({
            "eval_id":             len(results) + 1,
            "trial_id":            row["trial_id"],
            "prompt_id":           prompt_id,
            "provider":            "google",
            "model":               "gemma-3-4b-it",
            "model_version":       None,
            "temperature":         0,
            "max_tokens":          50,
            "response_larger":     parsed,
            "response_confidence": None,
            "raw_response":        response,
            "latency_ms":          latency_ms,
            "timestamp":           datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "error":               None
        })

print(f"Total evals: {len(results)}")

## Save & Summarize

In [ ]:
evals = pd.DataFrame(results)

# Model answers by tier
print(f"{len(evals)} evals recorded")
evals.merge(trials[["trial_id", "tier"]], on="trial_id").groupby("tier")["response_larger"].value_counts()

# Phase 3
## Analyze Results
Analysis: join evals with trial/prompt metadata, compute accuracy and bias metrics, and display all diagnostic plots inline.

This is somewhat of an optional phase. You are encourgaed to come up with your own plots and analysis!

In [ ]:
from src.analyze import analyze_results

results = analyze_results(
    show_plots=False,
    evals_df=evals,
    trials_path=TRIALS_CSV,
    prompts_path=f"{INPUT}/data/prompts.csv",
)